# Calibration

## 1 - Environnement

### 1.1 - Dépendances

In [ ]:
# pip

%pip install --upgrade pip

In [ ]:
# installation des dépendances

%pip install --upgrade sys
%pip install --upgrade os

%pip install --upgrade pathlib

%pip install --upgrade importlib

%pip install --upgrade numpy
%pip install --upgrade pandas

%pip install --upgrade shapely

%pip install --upgrade ast

%pip install --upgrade logging

In [ ]:
# chargement des dépendances

import sys
import os

from pathlib import Path

import importlib

import utm

import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import geopandas as gpd

from shapely import wkt
from shapely.geometry import Point
from shapely.geometry import Polygon
from shapely.ops import unary_union

import contextily as ctx


import ast

import logging

In [ ]:
# configuration du logging

logging.basicConfig(
    level=logging.INFO,  
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

In [ ]:
# chemin relatif depuis ce notebook jusqu'au dossier parent de modules 

project_root = Path.cwd().parents[3]
sys.path.insert(0, str(project_root))

### 1.2 - Classes et fonctions

In [ ]:
# import des classes

import modules.Utils as Utils

importlib.reload(Utils)

In [ ]:
# fonction de gestion des formats de colonnes avec horaires

def process_time_variable(df, column="DEPARTURE"):
    """
    Extract the time (HH:MM:SS) from a string, converts in datetime.
    Round to the minute.


    Parameters
    ----------
    df : pandas.DataFrame
        The DataFrame whose column to convert.
    column : str
        Name of the column to comvert

    Returns
    -------
    pandas.DataFrame
        The transformed DataFrame
    """
    def replace_24_hour(t):
        if t.startswith("24:"):
            return "00:" + t.split(":", 1)[1]
        return t
    
    df[column] = df[column].apply(replace_24_hour)
    df[column] = df[column].str.extract(r'(\d{2}:\d{2}:\d{2})')[0]
    df[column] = pd.to_datetime(df[column], format='%H:%M:%S', errors='coerce')
    df[column] = df[column].dt.round("min")
    return df

def restore_time_variable(df, column="DEPARTURE"):
    """
    Convert a datetime column back to a string time format HH:MM:SS.
    Restore '24:' prefix if the original time was midnight after rounding.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing the time column.
    column : str
        Name of the column to convert.

    Returns
    -------
    pandas.DataFrame
        The transformed DataFrame with the column as string HH:MM:SS.
    """

    if column not in df.columns:
        raise ValueError(f"Column '{column}' not found in DataFrame.")

    # Round to nearest minute
    df[column] = df[column].dt.round("min")

    # Convert to string HH:MM:SS
    df[column] = df[column].dt.strftime("%H:%M:%S")

    def restore_24_format(t):
        # If time is exactly midnight "00:xx:yy" AND original was "24:xx:yy"
        # → We cannot detect original directly anymore, so we only restore pure midnight
        if t.startswith("00:") and t != "00:00:00":
            return "24:" + t.split(":", 1)[1]
        return t

    df[column] = df[column].apply(restore_24_format)

    return df


In [ ]:
# fonction calibration

def calibrate(df, target_number = 0, random_state=42):
    """
    Duplicate or remove lines so the number of lines is equal to target_number

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame to modify.
    target_number : int
        target number of lines.
    random_state : int
        Random state.
    Return
    -------
    pandas.DataFrame
        The modified dataframe.
    """

    # check
    if df.empty:
        raise ValueError("Null or empty dataframe.")
    if target_number <= 0:
        raise ValueError("Invalid target number")



    variation_df = df.copy()
    variation_df = process_time_variable(variation_df, "DEPARTURE")
    
    # if delta = 0 → nothing to do
    delta = target_number - len(df)

    if delta == 0:
        variation_df["ID"] = list(range(len(variation_df)))
        variation_df = restore_time_variable(variation_df, "DEPARTURE")
        return variation_df.copy()


    # sampling
    if delta > 0:
        # duplication
        sampled = variation_df.sample(
            n=delta,
            replace=True,
            random_state=random_state
        )
        variation_df = pd.concat([variation_df, sampled], ignore_index=True)

    else:
        # deletion
        n_remove = -delta

        if n_remove >= len(variation_df):
            raise ValueError(
                f"Cannot remove {n_remove} rows: period only contains {len(variation_df)} rows."
            )

        to_remove = variation_df.sample(
            n=n_remove,
            replace=False,
            random_state=random_state
        ).index

        variation_df = variation_df.drop(index=to_remove).reset_index(drop=True)

    # renumber IDs

    variation_df["ID"] = list(range(len(variation_df)))
    variation_df = restore_time_variable(variation_df, "DEPARTURE")
    return variation_df.copy()
    

In [ ]:
# point aléatoire dans un polygon

def random_point_in_polygon(poly):
    minx, miny, maxx, maxy = poly.bounds
    while True:
        x = np.random.uniform(minx, maxx)
        y = np.random.uniform(miny, maxy)
        p = Point(float(x), float(y))
        if p.within(poly):
            return p

In [ ]:
# fonction réparition spatiale




def spatial_spread_amsterdam(df, threshold=30):
    """
    Return a demand df with origin and destination points randomly spread in their spatial zone,
    except those whose point values are part of the excluded set.
    """

    if df.empty:
        raise ValueError("Null or empty dataframe.")

    df = df.copy()
    o, _ = Utils.prepare_gdf(df, "epsg:32631")
    # 1 — ORIGINS à exclure
    excl = (
        o.groupby("ORIGIN")["ID"]
          .count()
          .reset_index()
    )
    excl = excl.loc[excl["ID"] > threshold, "ORIGIN"]
    excl = set(excl)

    # 2 — Charger zones
    #demand_areas = gpd.read_file("../../data/AmsterdamCity/demand_areas/demand_areas.shp")
    demand_areas = gpd.read_file("../../data/AmsterdamCity/Amsterdam_inner/Amsterdam_inner.shp")
    demand_areas = demand_areas.rename(columns={"[('Deletio": "ZONE_ID", "geometry": "GEOMETRY"})
    demand_areas["ZONE_ID"] = demand_areas["ZONE_ID"].astype(int)
    demand_areas = gpd.GeoDataFrame(demand_areas, geometry="GEOMETRY", crs="epsg:28992")
    demand_areas = demand_areas.to_crs("epsg:32631")

    # 3 — convertir en GeoDataFrame (comme avant)
    origin_gdf, destination_gdf = Utils.prepare_gdf(df, "epsg:32631")
    df = origin_gdf.copy()
    df["DESTINATION"] = destination_gdf["DESTINATION"]

    # 4 — Zones des ORIGIN
    df = gpd.GeoDataFrame(df, geometry="ORIGIN", crs="epsg:32631")
    df["ORIGIN_ZONE"] = df.set_geometry("ORIGIN") \
                          .sjoin(demand_areas, how="left", predicate="within")["ZONE_ID"]

    # 5 — Zones des DESTINATION
    df = gpd.GeoDataFrame(df, geometry="DESTINATION", crs="epsg:32631")
    df["DESTINATION_ZONE"] = df.set_geometry("DESTINATION") \
                               .sjoin(demand_areas, how="left", predicate="within")["ZONE_ID"]

    # dict zone -> polygon
    demand_areas_dict = demand_areas.set_index("ZONE_ID")["GEOMETRY"].to_dict()

    # 6 — Modifier ORIGIN uniquement si ORIGIN ∉ excl
    def modify_origin(row):
        if row["ORIGIN"] in excl:    # garder tel quel
            return row["ORIGIN"]
        return random_point_in_polygon(demand_areas_dict[row["ORIGIN_ZONE"]])

    # 7 — Modifier DESTINATION uniquement si DESTINATION ∉ excl
    def modify_destination(row):
        if row["DESTINATION"] in excl:
            return row["DESTINATION"]
        return random_point_in_polygon(demand_areas_dict[row["DESTINATION_ZONE"]])

    df["ORIGIN"] = df.apply(modify_origin, axis=1)
    df["DESTINATION"] = df.apply(modify_destination, axis=1)

    # Format final
    df["ORIGIN"] = df["ORIGIN"].apply(lambda p: f"{p.x} {p.y}")
    df["DESTINATION"] = df["DESTINATION"].apply(lambda p: f"{p.x} {p.y}")

    df = df.sort_values("DEPARTURE")
    df = df.drop(columns=["SERVICE"])
    df["SERVICE"] = "PersonalVehicle"

    return df.drop(columns=["ORIGIN_ZONE", "DESTINATION_ZONE"], errors="ignore").copy()



In [ ]:
# fonction de discrimination des trajets en dehors de la zone d'étude et d'association de points d'entrées ou de sorties
# aux trajets possédants une extrémitée en dehors de la zone d'étude

def nearest_highway(highway_latlon, point_lat, point_lon):
    """
    Find the nearest highway point to a given lat/lon point.
    
    highway_latlon: list of [lat, lon]
    point_lat, point_lon: coordinates in lat/lon
    Returns nearest highway point (lat, lon)
    """
    dist = [(point_lat - h[0])**2 + (point_lon - h[1])**2 for h in highway_latlon]
    min_idx = dist.index(min(dist))
    return highway_latlon[min_idx]


def all_in_city(df, global_demand_area):
    """
    Adjust ORIGIN/DESTINATION points:
    - If one point is outside the city area, project it on the nearest highway point.
    - If both points are outside, remove the row.
    
    df contains ORIGIN/DESTINATION in EPSG:32631 (x y as strings "x y").
    global_demand_area is Polygon/MultiPolygon in EPSG:32631.
    highway points are in EPSG:4326 (lat, lon).
    """
    
    # Highway reference points (lat, lon)
    highway = [
        [52.32791994290011, 4.913239361372001],
        [52.348521124104536, 4.969982909959437],
        [52.38911039306189, 4.967560455680146],
        [52.40771996733094, 4.938153295847306],
        [52.4233956111038, 4.877273846428637],
        [52.3960595030744, 4.844561046121149],
        [52.3841084139694, 4.8444926286081955],
        [52.338567002294766, 4.841697415785518],
    ]

    list_long = [4.875118,
            4.939276,
            4.972082,
            4.962623,
            4.942704,
            4.913261,
            4.842213,
            4.844446]
    list_lat = [52.425440,
            52.407426,
            52.367443,
            52.351031,
            52.336912,
            52.328299,
            52.338848,
            52.396390]

    # Liste combinée [lat, lon]
    highway = [[lat, lon] for lat, lon in zip(list_lat, list_long)]
    
    # Convert ORIGIN/DESTINATION (EPSG:32631) to shapely Points
    df["ORIGIN_geom"] = df["ORIGIN"].apply(lambda s: Point(*map(float, s.split())))
    df["DEST_geom"] = df["DESTINATION"].apply(lambda s: Point(*map(float, s.split())))

    demand_areas = gpd.read_file("../../data/AmsterdamCity/Amsterdam_inner/Amsterdam_inner.shp")
    demand_areas = demand_areas.rename(columns={"[('Deletio": "ZONE_ID", "geometry": "GEOMETRY"})
    demand_areas["ZONE_ID"] = demand_areas["ZONE_ID"].astype(int)
    demand_areas = gpd.GeoDataFrame(demand_areas, geometry="GEOMETRY", crs="epsg:28992")
    demand_areas = demand_areas.to_crs("epsg:32631")

    # 3 — convertir en GeoDataFrame (comme avant)
    origin_gdf, destination_gdf = Utils.prepare_gdf(df, "epsg:32631")
    df = origin_gdf.copy()
    df["DESTINATION"] = destination_gdf["DESTINATION"]

    # 4 — Zones des ORIGIN
    df = gpd.GeoDataFrame(df, geometry="ORIGIN", crs="epsg:32631")
    df["ORIGIN_ZONE"] = df.set_geometry("ORIGIN") \
                          .sjoin(demand_areas, how="left", predicate="within")["ZONE_ID"]

    # 5 — Zones des DESTINATION
    df = gpd.GeoDataFrame(df, geometry="DESTINATION", crs="epsg:32631")
    df["DESTINATION_ZONE"] = df.set_geometry("DESTINATION") \
                               .sjoin(demand_areas, how="left", predicate="within")["ZONE_ID"]
    
    new_rows = []

    for idx, row in df.iterrows():
        o = row["ORIGIN_geom"]
        d = row["DEST_geom"]

        o_in = o.within(global_demand_area)
        d_in = d.within(global_demand_area)

        # Case 1: both outside → skip
        if not o_in and not d_in:
            continue

        # Case 2: ORIGIN inside, DEST outside → project DEST
        if o_in and not d_in:
            lat, lon = utm.to_latlon(d.x, d.y, 31, 'N')
            lat2, lon2 = nearest_highway(highway, lat, lon)
            x, y, _, _ = utm.from_latlon(lat2, lon2)
            d = Point(x, y)

        # Case 3: ORIGIN outside, DEST inside → project ORIGIN
        elif not o_in and d_in:
            lat, lon = utm.to_latlon(o.x, o.y, 31, 'N')
            lat2, lon2 = nearest_highway(highway, lat, lon)
            x, y, _, _ = utm.from_latlon(lat2, lon2)
            o = Point(x, y)

        new_rows.append({
            "ID": row.ID,
            "DEPARTURE": row.DEPARTURE,
            "ORIGIN": f"{o.x} {o.y}",
            "DESTINATION": f"{d.x} {d.y}",
            "SERVICE":row.SERVICE
        })

    return pd.DataFrame(new_rows)


def all_in_city(df, global_demand_area):
    """
    Improved version:
    - Points considered inside the city only if geometrically inside AND their *_ZONE is not NaN.
    - Missing-zone points (NaN zones) are treated as outside.
    - Outside points are projected to nearest highway point.
    - If both ORIGIN and DESTINATION are outside → row removed.
    """

    # Highway reference points
    list_long = [4.875118, 4.939276, 4.972082, 4.962623, 4.942704, 4.913261, 4.842213, 4.844446]
    list_lat  = [52.425440, 52.407426, 52.367443, 52.351031, 52.336912, 52.328299, 52.338848, 52.396390]
    highway = [[lat, lon] for lat, lon in zip(list_lat, list_long)]

    # Convert ORIGIN/DEST to shapely Points
    df["ORIGIN_geom"] = df["ORIGIN"].apply(lambda s: Point(*map(float, s.split())))
    df["DEST_geom"] = df["DESTINATION"].apply(lambda s: Point(*map(float, s.split())))

    demand_areas = gpd.read_file("../../data/AmsterdamCity/Amsterdam_inner/Amsterdam_inner.shp")
    demand_areas = demand_areas.rename(columns={"[('Deletio": "ZONE_ID", "geometry": "GEOMETRY"})
    demand_areas["ZONE_ID"] = demand_areas["ZONE_ID"].astype(int)
    demand_areas = gpd.GeoDataFrame(demand_areas, geometry="GEOMETRY", crs="epsg:28992")
    demand_areas = demand_areas.to_crs("epsg:32631")

    # 3 — convertir en GeoDataFrame (comme avant)
    origin_gdf, destination_gdf = Utils.prepare_gdf(df, "epsg:32631")
    df = origin_gdf.copy()
    df["DESTINATION"] = destination_gdf["DESTINATION"]

    # 4 — Zones des ORIGIN
    df = gpd.GeoDataFrame(df, geometry="ORIGIN", crs="epsg:32631")
    df["ORIGIN_ZONE"] = df.set_geometry("ORIGIN") \
                          .sjoin(demand_areas, how="left", predicate="within")["ZONE_ID"]

    # 5 — Zones des DESTINATION
    df = gpd.GeoDataFrame(df, geometry="DESTINATION", crs="epsg:32631")
    df["DESTINATION_ZONE"] = df.set_geometry("DESTINATION") \
                               .sjoin(demand_areas, how="left", predicate="within")["ZONE_ID"]

    new_rows = []

    for idx, row in df.iterrows():
        o = row["ORIGIN_geom"]
        d = row["DEST_geom"]

        # -----------------------------
        # NEW → treat NaN zones as OUTSIDE
        # -----------------------------
        origin_zone_ok = not pd.isna(row["ORIGIN_ZONE"])
        dest_zone_ok   = not pd.isna(row["DESTINATION_ZONE"])

        o_in_geom = o.within(global_demand_area)
        d_in_geom = d.within(global_demand_area)

        # Final inside/outside evaluation
        o_in = o_in_geom and origin_zone_ok
        d_in = d_in_geom and dest_zone_ok

        # Case 1 → both outside → skip entirely
        if not o_in and not d_in:
            continue

        # Case 2 → ORIGIN inside, DEST outside → project DEST
        if o_in and not d_in:
            lat, lon = utm.to_latlon(d.x, d.y, 31, 'N')
            lat2, lon2 = nearest_highway(highway, lat, lon)
            x, y, _, _ = utm.from_latlon(lat2, lon2)
            d = Point(x, y)

        # Case 3 → ORIGIN outside, DEST inside → project ORIGIN
        elif not o_in and d_in:
            lat, lon = utm.to_latlon(o.x, o.y, 31, 'N')
            lat2, lon2 = nearest_highway(highway, lat, lon)
            x, y, _, _ = utm.from_latlon(lat2, lon2)
            o = Point(x, y)

        new_rows.append({
            "ID": row.ID,
            "DEPARTURE": row.DEPARTURE,
            "ORIGIN": f"{o.x} {o.y}",
            "DESTINATION": f"{d.x} {d.y}",
            "SERVICE": row.SERVICE
        })

    return pd.DataFrame(new_rows)


## 2 - Contexte, chargement, prétraitement

### 2.1 - Contexte

Dans le cadre d'un travail d'analyse de sensibilité aux émissions de méthodes de construction de demande fichiers d'entrées du simulateur MnMS, il est essentiel, malgré les différences d'hypothèses, de considérer des demandes aux effectifs totaux comparables. En cas de différences trop importantes, il est essentiel de relacibrer les effectifs sur totaux sur une référence. Ici, ce travail est réalisé sur deux demandes aux origines disparates : la première est produit de fichiers de sorties de simulation MATSim et de travaux de modélisations d'activités aux Pays-bas, la deuxième est issue de fichiers fournies par la municipalité d'Amsterdam et fera office de référence.
Se référer au notebook *Demandes* pour plus de détail à propos des constructions des deux fichiers de demande.

### 2.2 - Chargement et prétraitement

In [ ]:
# chargement des demandes

chemin_demande_matsim = "../../inputs/Matsim_7-9/Matsim_7-9_Original__Original/Matsim_7-9_Original__Original__None_None__None_None__None__0.00__0_0.00__0.00.csv"
demande_matsim = pd.read_csv(chemin_demande_matsim, sep=';')

chemin_demande_amsterdam = "../../inputs/AmsterdamCity/AmsterdamCity_Original__Original/AmsterdamCity_Original__Original__None_None__None_None__None__0.00__0_0.00__0.00.csv"
demande_amsterdam = pd.read_csv(chemin_demande_amsterdam, sep=';') 

In [ ]:
# zone de la demande

demand_area = gpd.read_file('../../data/AmsterdamCity/Amsterdam_inner/Amsterdam_inner.shp')
demand_area = gpd.GeoDataFrame(demand_area, geometry='geometry', crs="epsg:28992")
demand_area = demand_area.to_crs("epsg:32631")
global_demand_area = unary_union(demand_area.geometry)

## 3 - Calibration

methode : tirage aléatoire uniforme et association a decoupage spatiale puis redistribution dans l'espace  et restriction a la zone de demande

In [ ]:
# Restriction à la zone de demande 

buffer = all_in_city(demande_matsim, global_demand_area)

In [ ]:
# Difference d'effectif total

print(f"Effectif total demande MATSim : {len(demande_matsim)}.")
print(f"Effectif total demande AmsterdamCity : {len(demande_amsterdam)}.")

In [ ]:
# calibration

buffer = calibrate(buffer, len(demande_amsterdam), random_state=42)

In [ ]:
# Difference d'effectif total

print(f"Effectif total demande MATSim : {len(buffer)}.")
print(f"Effectif total demande AmsterdamCity : {len(demande_amsterdam)}.")

In [ ]:
# repartition des points de manière aléatoire dans une zone associée du découpage géographique d'Amsterdam par code postal

buffer = spatial_spread_amsterdam(buffer, 50)

In [ ]:
# sauvegarde 

buffer.to_csv("../../inputs/Matsim_7-9/Matsim_7-9_Calibration__Original/Matsim_7-9_Calibration__Original__None_None__None_None__None__0.00__0_0.00__0.00.csv", sep=';', index=False)
chemin_demande_matsim_calibre = "../../inputs/Matsim_7-9/Matsim_7-9_Calibration__Original/Matsim_7-9_Calibration__Original__None_None__None_None__None__0.00__0_0.00__0.00.csv"

In [ ]:
# affichage des origines de la demande calibrée 

origines_demande_amsterdam, destinations_demande_amsterdam = Utils.prepare_gdf(buffer, "EPSG:32631")
ori_sum = origines_demande_amsterdam.groupby("ORIGIN")["ID"].count()
dest_sum = destinations_demande_amsterdam.groupby("DESTINATION")["ID"].count()
max_size_value = max(ori_sum.max(), dest_sum.max())
Utils.display_origin_time_distribution(chemin_demande_matsim_calibre, "Demande calibrée MATSim : Distribution spatio-temporelle des origines", crs= "EPSG:32631",  interval="10min", max_size=max_size_value)

In [ ]:
# affichage des destinations de la demande calibrée 

Utils.display_destination_time_distribution(chemin_demande_matsim_calibre, "Demande calibrée MATSim : Distribution spatio-temporelle des origines", crs= "EPSG:32631",  interval="10min", max_size=max_size_value)


Les demandes étant maintenant comparable, plusieurs simulations sont réalisées pour chacune d'entre elles.

In [ ]:
Certaines origines et destinations sont clairement à l'extieure de la zone d'étude, on va l'

In [ ]:
demande_amsterdam

In [ ]:
demand_area = gpd.read_file('../../data/AmsterdamCity/Amsterdam_inner/Amsterdam_inner.shp')
demand_area.plot()

In [ ]:

demand_areas = gpd.read_file("../../data/AmsterdamCity/Amsterdam_inner/Amsterdam_inner.shp")
demand_areas = demand_areas.rename(columns={"[('Deletio": "ZONE_ID", "geometry": "GEOMETRY"})
demand_areas["ZONE_ID"] = demand_areas["ZONE_ID"].astype(int)
demand_areas = gpd.GeoDataFrame(demand_areas, geometry="GEOMETRY", crs="epsg:28992")
demand_areas = demand_areas.to_crs("epsg:32631")
origin_gdf, destination_gdf = Utils.prepare_gdf(buffer, "epsg:32631")
df = origin_gdf.copy()
df["DESTINATION"] = destination_gdf["DESTINATION"]

df = df.copy()
o, _ = Utils.prepare_gdf(df, "epsg:32631")
# 1 — ORIGINS à exclure
excl = (
    o.groupby("ORIGIN")["ID"]
        .count()
        .reset_index()
)
excl = excl.loc[excl["ID"] > 30, "ORIGIN"]
excl = set(excl)

# 4 — Zones des ORIGIN
df = gpd.GeoDataFrame(df, geometry="ORIGIN", crs="epsg:32631")
df["ORIGIN_ZONE"] = df.set_geometry("ORIGIN") \
                        .sjoin(demand_areas, how="left", predicate="within")["ZONE_ID"]

In [ ]:
df.isna().sum()

In [ ]:
nan_idx = df[df["ORIGIN_ZONE"].isna()].index
subset = df.loc[nan_idx]

# ⚠️ 3. IMPORTANT : reprojeter en Web Mercator (EPSG:3857) pour afficher un fond de carte
gdf2_3857 = df.to_crs(3857)
subset_3857 = subset.to_crs(3857)

# 4. Affichage avec fond OSM
fig, ax = plt.subplots(figsize=(10, 10))

# couche globale
gdf2_3857.plot(
    ax=ax,
    color="lightgrey",
    edgecolor="white",
    linewidth=0.5
)

# les objets NaN en rouge
subset_3857.plot(
    ax=ax,
    edgecolor="red",
    facecolor="none",
    linewidth=2
)

# Ajouter le fond de carte
ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)

ax.set_title("Objets correspondant aux NaN dans ORIGIN_ZONE")
plt.tight_layout()
plt.show()


In [ ]:
mask = df["ORIGIN"].isin(excl)
gdf_excl = df[mask]
idx_excl = df[mask].index
import contextily as ctx
import matplotlib.pyplot as plt

gdf_3857 = df.to_crs(3857)
gdf_excl_3857 = gdf_excl.to_crs(3857)

fig, ax = plt.subplots(figsize=(10,10))

# Tous les points
gdf_3857.plot(ax=ax, color="grey", markersize=5)

# Les points appartenant à excl
gdf_excl_3857.plot(ax=ax, color="red", markersize=20)

ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)

ax.set_title("Points appartenant à excl")
plt.show()



In [ ]:
excl

In [ ]:
subset.sample(5)